# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 04: Model Training & Artifact Generation
# ==========================================================

Objective
---------
1. Load the processed feature dataset generated in Notebook 03.
2. Split the dataset into training and testing sets.
3. Compute class weights to handle class imbalance.
4. Build Apache Spark preprocessing pipelines.
5. Train multiple machine learning models.
6. Save trained models, preprocessing pipelines, and testing dataset.
7. Generate baseline evaluation metrics for further analysis.

Mục tiêu
--------
1. Nạp tập dữ liệu đặc trưng được tạo từ Notebook 03.
2. Chia dữ liệu thành tập huấn luyện và kiểm thử.
3. Tính trọng số lớp để xử lý mất cân bằng dữ liệu.
4. Xây dựng Pipeline tiền xử lý bằng Apache Spark.
5. Huấn luyện nhiều mô hình học máy.
6. Lưu mô hình, Pipeline và tập kiểm thử.
7. Tạo kết quả đánh giá ban đầu phục vụ Notebook 05.
"""

1. THIẾT LẬP VÀ KHẢO SÁT BAN ĐẦU

In [1]:
import os

print(os.listdir("../data/processed"))

['.gitkeep', 'seer_breast_cancer_clean', 'seer_breast_cancer_feature', 'test_dataset.csv', 'test_dataset.parquet']


In [2]:
import shutil
import os

paths = [
    "../models/trained_models/logistic_regression_pipeline",
    "../models/trained_models/tree_models_pipeline",
    "../models/trained_models/logistic_regression_model",
    "../models/trained_models/decision_tree_model",
    "../models/trained_models/random_forest_model",
    "../models/trained_models/gbt_classifier_model"
]

for path in paths:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Removed: {path}")

print("Old artifacts cleaned.")

Removed: ../models/trained_models/logistic_regression_pipeline
Removed: ../models/trained_models/tree_models_pipeline
Removed: ../models/trained_models/logistic_regression_model
Removed: ../models/trained_models/decision_tree_model
Removed: ../models/trained_models/random_forest_model
Removed: ../models/trained_models/gbt_classifier_model
Old artifacts cleaned.


In [3]:
# 1. Import Libraries | Khai báo thư viện

print("=" * 60)
print("1. IMPORT LIBRARIES")
print("=" * 60)

import os 
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Add project root directory to path | Thêm thư mục gốc dự án vào hệ thống
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions from src | Nạp các hàm tự định nghĩa từ src
from src.data.loader import create_spark_session, load_csv

1. IMPORT LIBRARIES


In [4]:
# 2. Create Spark Session | Khởi tạo Spark Session

print("=" * 60)
print("2. CREATE SPARK SESSION")
print("=" * 60)

spark = create_spark_session("SEER Breast Cancer Model Training")

2. CREATE SPARK SESSION


In [5]:
# 3. Load Feature Dataset | Tải dữ liệu

print("\n" + "=" * 60)
print("3. LOAD FEATURE DATASET")
print("=" * 60)

path = os.path.abspath("../data/processed/seer_breast_cancer_feature")
df = spark.read.parquet(path)

print(f"Dataset loaded successfully. Rows: {df.count():,}")


3. LOAD FEATURE DATASET
Dataset loaded successfully. Rows: 456,087


In [6]:
# 4: Dataset Overview & Schema | Khảo sát cấu trúc và lược đồ dữ liệu

print("\n" + "=" * 60)
print("4. DATASET OVERVIEW & SCHEMA")
print("=" * 60)

print(f"Total Rows    : {df.count():,}")
print(f"Total Columns : {len(df.columns)}")
df.printSchema()


4. DATASET OVERVIEW & SCHEMA
Total Rows    : 456,087
Total Columns : 26
root
 |-- Age: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: integer (nullable = true)
 |-- Grade: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: integer (nullable = true)
 |-- Regional_Nodes_Positive: integer (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: string (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: string (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = true)
 |-- Radiation: string (nullable = true)
 |-- Chemotherapy: string (nullable = true)
 |-- AJCC_Stage: string 

In [7]:
# 5: Preview Dataset | Hiển thị mẫu dữ liệu thực tế

print("\n" + "=" * 60)
print("5. PREVIEW DATASET")
print("=" * 60)
df.show(5, truncate=False)


5. PREVIEW DATASET
+----+------+-------------------------+------------------------------+----------+-----------------------------------+------+------+-----------------------+-----------------------+----------------+---------------+-------------------------+-----------------------+------+--------------------+--------------------------+-------------------------------------------------------------------------+--------------+------------+----------+-----+------------+----------------+-------------------+--------------+
|Age |Sex   |Race                     |Marital_Status                |Tumor_Size|Grade                              |AJCC_T|AJCC_N|Regional_Nodes_Examined|Regional_Nodes_Positive|Sequence_Number |Histologic_Type|Laterality               |Diagnostic_Confirmation|AJCC_M|Surgery_Primary_Site|Surgery_Other_Regional    |Surgery_Radiation_Sequence                                               |Radiation     |Chemotherapy|AJCC_Stage|label|Age_Group   |Tumor_Size_Group|Node_Ratio  

2. PHÂN CHIA DỮ LIỆU & XỬ LÝ MẤT CÂN BẰNG (CHẶN DATA LEAKAGE)

In [8]:
# 6: Train/Test Split | Thực hiện phân chia dữ liệu huấn luyện/kiểm thử

print("\n" + "=" * 60)
print("6. TRAIN/TEST SPLIT (PREVENT DATA LEAKAGE)")
print("=" * 60)

# Secure a reproducible split using random seed | Cố định phân vùng bằng seed trước khi xử lý pipeline để chặn rò rỉ thông tin
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Dataset Rows : {train_df.count():,}")
print(f"Testing Dataset Rows  : {test_df.count():,}")


6. TRAIN/TEST SPLIT (PREVENT DATA LEAKAGE)
Training Dataset Rows : 365,331
Testing Dataset Rows  : 90,756


In [9]:
# 7: Check Class Distribution | Thống kê tỷ lệ nhãn Alive/Dead trên tập Train

print("\n" + "=" * 60)
print("7. CHECK CLASS DISTRIBUTION")
print("=" * 60)

class_counts = train_df.groupBy("label").count().collect()
counts = {row["label"]: row["count"] for row in class_counts}
print(f"Class distributions in Train Set: {counts}")


7. CHECK CLASS DISTRIBUTION
Class distributions in Train Set: {1: 135199, 0: 230132}


In [10]:
# 8: Compute Class Weights | Tính toán trọng số nghịch đảo và gán cột 'weight' cho tập Train

print("\n" + "=" * 60)
print("8. COMPUTE CLASS WEIGHTS")
print("=" * 60)

total_count = train_df.count()
weight_0 = total_count / (2.0 * counts[0.0])
weight_1 = total_count / (2.0 * counts[1.0])

print(f"Calculated class weights: Class 0 (Alive) = {weight_0:.4f} | Class 1 (Dead) = {weight_1:.4f}")

# Map weights into a new train column | Đổ cột trọng số thực tế vào tập Train
train_df = train_df.withColumn(
    "weight",
    F.when(F.col("label") == 1.0, weight_1).otherwise(weight_0)
)

# Test set has no weights to prevent leakage | Không gán trọng số cho tập Test nhằm chặn rò rỉ dữ liệu
test_df = test_df.withColumn("weight", F.lit(1.0))


8. COMPUTE CLASS WEIGHTS
Calculated class weights: Class 0 (Alive) = 0.7937 | Class 1 (Dead) = 1.3511


3. THIẾT LẬP PIPELINE SONG SONG (TỐI ƯU HÓA CHO SPARK CLUSTER)

In [11]:
# Define categorical and numerical features | Khai báo nhóm thuộc tính đặc trưng
# [CORRECTED] Sequence_Number removed as it was identified as a constant feature in Notebook 02
categorical_cols = [
    "Sex", "Race", "Marital_Status", "Grade", "AJCC_T", "AJCC_N", "AJCC_M", 
    "AJCC_Stage", "Laterality", "Diagnostic_Confirmation", 
    "Surgery_Other_Regional", "Surgery_Radiation_Sequence", "Radiation", 
    "Chemotherapy", "Age_Group", "Tumor_Size_Group", "Hormone_Status",
    "Histologic_Type", "Surgery_Primary_Site"
]

numerical_cols = [
    "Age", "Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive", "Node_Ratio"
]

In [12]:
# 9: StringIndexer Setup | Mã hóa nhãn phân loại thành số nguyên

print("\n" + "=" * 60)
print("9. STRINGINDEXER SETUP")
print("=" * 60)

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_cols
]
indexed_categorical_cols = [f"{col}_indexed" for col in categorical_cols]
print(f"Created StringIndexers for {len(categorical_cols)} categorical features.")


9. STRINGINDEXER SETUP
Created StringIndexers for 19 categorical features.


In [13]:
# 10: OneHotEncoder Setup | Mã hóa One-Hot từ cột Index (chỉ cho mô hình tuyến tính)

print("\n" + "=" * 60)
print("10. ONEHOTENCODER SETUP")
print("=" * 60)

encoder = OneHotEncoder(
    inputCols=indexed_categorical_cols,
    outputCols=[f"{col}_encoded" for col in categorical_cols], handleInvalid="keep"
)
encoded_categorical_cols = [f"{col}_encoded" for col in categorical_cols]
print("OneHotEncoder configured for linear models.")


10. ONEHOTENCODER SETUP
OneHotEncoder configured for linear models.


In [14]:
# 11: VectorAssembler Setup | Thiết lập 2 bộ Assembler độc lập

print("\n" + "=" * 60)
print("11. VECTORASSEMBLER SETUP")
print("=" * 60)

# Assembler 1: Linear Models (One-Hot Categories + Scaled Numerics)
assembler_linear = VectorAssembler(
    inputCols=encoded_categorical_cols + numerical_cols,
    outputCol="unscaled_features",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

# Assembler 2: Tree Models (Raw Categorical Indexes + Raw Numerics to avoid dummy variables OOM)
assembler_tree = VectorAssembler(
    inputCols=indexed_categorical_cols + numerical_cols,
    outputCol="features"
)

print("Dual VectorAssembler successfully initialized.")


11. VECTORASSEMBLER SETUP
Dual VectorAssembler successfully initialized.


In [15]:
# 12: Transform Train & Test Sets | Thực thi Pipeline (Stable Version)

print("\n" + "=" * 60)
print("12. TRANSFORM TRAIN & TEST SETS (STABLE - NO CACHE)")
print("=" * 60)

# Build transformation pipelines | Biên dịch luồng Pipeline
pipeline_linear = Pipeline(stages=indexers + [encoder, assembler_linear, scaler])
pipeline_tree = Pipeline(stages=indexers + [assembler_tree])

print("Fitting transformation pipelines on Train set...")
preproc_model_linear = pipeline_linear.fit(train_df)
preproc_model_tree = pipeline_tree.fit(train_df)

# Generate final transformed datasets | Thực thi chuyển đổi cấu trúc
# Spark sẽ thực hiện các biến đổi này mỗi khi dữ liệu được gọi tới (Lazy Evaluation)

train_linear = preproc_model_linear.transform(train_df)
test_linear = preproc_model_linear.transform(test_df)

train_tree = preproc_model_tree.transform(train_df)
test_tree = preproc_model_tree.transform(test_df)

print("Transformation completed successfully.")
print(f"Ready to proceed with modeling using train_linear and train_tree.")


12. TRANSFORM TRAIN & TEST SETS (STABLE - NO CACHE)
Fitting transformation pipelines on Train set...
Transformation completed successfully.
Ready to proceed with modeling using train_linear and train_tree.


4. HUẤN LUYỆN MÔ HÌNH VÀ THỰC NGHIỆM ĐO THỜI GIAN

In [16]:
training_times = {}

In [17]:
# 13: Train Logistic Regression | Huấn luyện mô hình Logistic Regression

print("\n" + "=" * 60)
print("13. TRAIN LOGISTIC REGRESSION")
print("=" * 60)

lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="weight", maxIter=100)
start_time = time.time()
lr_model = lr.fit(train_linear)
training_times["Logistic Regression"] = time.time() - start_time
print(f"Logistic Regression trained in {training_times['Logistic Regression']:.2f} seconds.")


13. TRAIN LOGISTIC REGRESSION
Logistic Regression trained in 54.03 seconds.


In [18]:
# 14: Train Decision Tree | Huấn luyện mô hình Decision Tree

print("\n" + "=" * 60)
print("14. TRAIN DECISION TREE")
print("=" * 60)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    seed=42,
    maxDepth=8,
    minInstancesPerNode=20,
    maxBins=150
)
start_time = time.time()
dt_model = dt.fit(train_tree)
training_times["Decision Tree"] = time.time() - start_time
print(f"Decision Tree trained in {training_times['Decision Tree']:.2f} seconds.")


14. TRAIN DECISION TREE
Decision Tree trained in 15.09 seconds.


In [19]:
# 15: Train Random Forest | Huấn luyện mô hình Random Forest

print("\n" + "=" * 60)
print("15. TRAIN RANDOM FOREST")
print("=" * 60)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42, numTrees=100, maxBins=150)
start_time = time.time()
rf_model = rf.fit(train_tree)
training_times["Random Forest"] = time.time() - start_time
print(f"Random Forest trained in {training_times['Random Forest']:.2f} seconds.")


15. TRAIN RANDOM FOREST
Random Forest trained in 21.39 seconds.


In [20]:
# 16: Train GBT Classifier | Huấn luyện mô hình GBT Classifier

print("\n" + "=" * 60)
print("16. TRAIN GBT CLASSIFIER")
print("=" * 60)

gbt = GBTClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42, maxIter=50, maxBins=150)
start_time = time.time()
gbt_model = gbt.fit(train_tree)
training_times["GBT Classifier"] = time.time() - start_time
print(f"GBT Classifier trained in {training_times['GBT Classifier']:.2f} seconds.")


16. TRAIN GBT CLASSIFIER
GBT Classifier trained in 82.89 seconds.


In [21]:
# 17: Compare Training Time | Xuất bảng so sánh thời gian thực thi của các mô hình trên Spark

print("\n" + "=" * 60)
print("17. COMPARE TRAINING TIME")
print("=" * 60)

print(f"{'Algorithm':<25} | {'Training Time (Seconds)':<25}")
print("-" * 55)
for name, elapsed in training_times.items():
    print(f"{name:<25} | {elapsed:<25.2f}")
print("-" * 55)


17. COMPARE TRAINING TIME
Algorithm                 | Training Time (Seconds)  
-------------------------------------------------------
Logistic Regression       | 54.03                    
Decision Tree             | 15.09                    
Random Forest             | 21.39                    
GBT Classifier            | 82.89                    
-------------------------------------------------------


5. DỰ ĐOÁN, ĐÓNG GÓI VÀ BÁO CÁO HỆ THỐNG

In [22]:
# 18: Generate Logistic Regression Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("18. GENERATE LOGISTIC REGRESSION PREDICTIONS")
print("=" * 60)
lr_predictions = lr_model.transform(test_linear)
print("Logistic Regression predictions generated.")


18. GENERATE LOGISTIC REGRESSION PREDICTIONS
Logistic Regression predictions generated.


In [23]:
#  19: Generate Decision Tree Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("19. GENERATE DECISION TREE PREDICTIONS")
print("=" * 60)
dt_predictions = dt_model.transform(test_tree)
print("Decision Tree predictions generated.")


19. GENERATE DECISION TREE PREDICTIONS
Decision Tree predictions generated.


In [24]:
# 20: Generate Random Forest Predictions | Dự đoán trên tập Test
print("\n" + "=" * 60)
print("20. GENERATE RANDOM FOREST PREDICTIONS")
print("=" * 60)
rf_predictions = rf_model.transform(test_tree)
print("Random Forest predictions generated.")


20. GENERATE RANDOM FOREST PREDICTIONS
Random Forest predictions generated.


In [25]:
# 21: Generate GBT Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("21. GENERATE GBT PREDICTIONS")
print("=" * 60)
gbt_predictions = gbt_model.transform(test_tree)
print("GBT Classifier predictions generated.")


21. GENERATE GBT PREDICTIONS
GBT Classifier predictions generated.


In [26]:
# 22: Validate Prediction Results | Kiểm tra tính toàn vẹn và khớp số lượng dòng dự đoán

print("\n" + "=" * 60)
print("22. VALIDATE PREDICTION RESULTS")
print("=" * 60)

test_row_count = test_df.count()
lr_count = lr_predictions.count()
dt_count = dt_predictions.count()
rf_count = rf_predictions.count()
gbt_count = gbt_predictions.count()

print(f"Target Testing Row Count : {test_row_count:,}")
print(f"Logistic Reg Predictions : {lr_count:,} | Status: {'MATCH' if lr_count == test_row_count else 'MISMATCH'}")
print(f"Decision Tree Predictions: {dt_count:,} | Status: {'MATCH' if dt_count == test_row_count else 'MISMATCH'}")
print(f"Random Forest Predictions: {rf_count:,} | Status: {'MATCH' if rf_count == test_row_count else 'MISMATCH'}")
print(f"GBT Classifier Preds     : {gbt_count:,} | Status: {'MATCH' if gbt_count == test_row_count else 'MISMATCH'}")


22. VALIDATE PREDICTION RESULTS
Target Testing Row Count : 90,756
Logistic Reg Predictions : 90,756 | Status: MATCH
Decision Tree Predictions: 90,756 | Status: MATCH
Random Forest Predictions: 90,756 | Status: MATCH
GBT Classifier Preds     : 90,756 | Status: MATCH


In [27]:
# 23. Save Trained Models, Pipelines, Test Dataset & Training Time | Lưu mô hình, Pipeline, tập kiểm thử và thời gian huấn luyện

import os
import shutil
import pandas as pd

print("\n" + "=" * 60)
print("23. SAVE TRAINED MODELS, PIPELINES, TEST DATASET & TRAINING TIME")
print("=" * 60)

# Create Output Directories | Tạo thư mục lưu kết quả

os.makedirs("../models/trained_models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports/tables", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

# 1. Save Preprocessing Pipelines | Lưu Pipeline tiền xử lý

print("\nSaving preprocessing pipelines...")
print("-" * 60)

pipeline_paths = {
    "logistic_regression_pipeline": preproc_model_linear,
    "tree_models_pipeline": preproc_model_tree
}

for name, model in pipeline_paths.items():

    path = os.path.abspath(f"../models/trained_models/{name}")

    if os.path.exists(path):
        shutil.rmtree(path)
    spark_path = "file:///" + path.replace("\\", "/")
    try:
        model.write().overwrite().save(spark_path)

        print(f"Pipeline saved : {name}")
        print(f"Location       : {path}")
    except Exception as e:
        print(f"Pipeline save failed : {name}")
        print(e)

# 2. Save Classification Models | Lưu các mô hình

print("\nSaving trained classification models...")
print("-" * 60)

models_to_save = {
    "logistic_regression_model": lr_model,
    "decision_tree_model": dt_model,
    "random_forest_model": rf_model,
    "gbt_classifier_model": gbt_model
}

for name, model in models_to_save.items():
    path = os.path.abspath(f"../models/trained_models/{name}")

    if os.path.exists(path):
        shutil.rmtree(path)
    spark_path = "file:///" + path.replace("\\", "/")

    try:
        model.write().overwrite().save(spark_path)
        print(f"Model saved    : {name}")
        print(f"Location       : {path}")
    except Exception as e:
        print(f"Model save failed : {name}")
        print(e)

# 3. Save Test Dataset | Lưu tập dữ liệu kiểm thử

print("\nSaving testing dataset...")
print("-" * 60)

test_path = "../data/processed/test_dataset.parquet"
test_df.write \
    .mode("overwrite") \
    .parquet(test_path)

print(f"Testing dataset saved: {test_path}")

# 4. Save Training Time | Lưu thời gian huấn luyện

print("\nSaving training time...")
print("-" * 60)

training_df = pd.DataFrame(list(training_times.items()),columns=["Model", "Training_Time"])

training_time_path = "../reports/tables/training_time.csv"

training_df.to_csv(training_time_path, index=False)

print(f"Training time saved: {training_time_path}")

print("\n" + "=" * 60)
print("ARTIFACT SAVING COMPLETED")
print("=" * 60)


23. SAVE TRAINED MODELS, PIPELINES, TEST DATASET & TRAINING TIME

Saving preprocessing pipelines...
------------------------------------------------------------
Pipeline saved : logistic_regression_pipeline
Location       : d:\Project_Breast_Cancer_SEER\models\trained_models\logistic_regression_pipeline
Pipeline saved : tree_models_pipeline
Location       : d:\Project_Breast_Cancer_SEER\models\trained_models\tree_models_pipeline

Saving trained classification models...
------------------------------------------------------------
Model saved    : logistic_regression_model
Location       : d:\Project_Breast_Cancer_SEER\models\trained_models\logistic_regression_model
Model saved    : decision_tree_model
Location       : d:\Project_Breast_Cancer_SEER\models\trained_models\decision_tree_model
Model saved    : random_forest_model
Location       : d:\Project_Breast_Cancer_SEER\models\trained_models\random_forest_model
Model saved    : gbt_classifier_model
Location       : d:\Project_Breast_C

In [28]:
# 24: Model Training Report & Baseline Validation

print("\n" + "=" * 60)
print("24. MODEL TRAINING REPORT & BASELINE VALIDATION")
print("=" * 60)

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

print("""
================================================================================
FINAL MODEL TRAINING PIPELINE COMPLETED
================================================================================
✓ Train/Test Split correctly isolated.
✓ Balanced class representation handled dynamically.
✓ Models trained successfully without memory-intensive caching.
""")

print(f"Logistic Regression Baseline ROC-AUC: {evaluator_auc.evaluate(lr_predictions):.4f}")
print(f"Decision Tree Baseline ROC-AUC      : {evaluator_auc.evaluate(dt_predictions):.4f}")
print(f"Random Forest Baseline ROC-AUC      : {evaluator_auc.evaluate(rf_predictions):.4f}")
print(f"GBT Classifier Baseline ROC-AUC     : {evaluator_auc.evaluate(gbt_predictions):.4f}")

print("\nReady for Notebook 05 — Advanced Model Evaluation!")


24. MODEL TRAINING REPORT & BASELINE VALIDATION

FINAL MODEL TRAINING PIPELINE COMPLETED
✓ Train/Test Split correctly isolated.
✓ Balanced class representation handled dynamically.
✓ Models trained successfully without memory-intensive caching.

Logistic Regression Baseline ROC-AUC: 0.8433
Decision Tree Baseline ROC-AUC      : 0.5532
Random Forest Baseline ROC-AUC      : 0.8326
GBT Classifier Baseline ROC-AUC     : 0.8560

Ready for Notebook 05 — Advanced Model Evaluation!
